# 第 8 章实验：XOR、非线性与反向传播
本实验只使用内存合成点。修改“参数”那一格并从头运行，观察隐藏层、学习率和线性模型的限制。

第一格会打印当前用的设备：有 NVIDIA 显卡并且装了带 CUDA 的 PyTorch 时是 `cuda`，否则是 `cpu`。这个实验很小，两种情况都只要几秒。最后一格会调用你在 `作业_手写两层网络.py` 里写的函数，作业还没完成时它只打印提示。

In [ ]:
import importlib.util
import os
from pathlib import Path

import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

# 图里的中文：Windows 自带的字体放在前面，找不到再试 macOS 和 Linux 上的
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Heiti SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# 设备：有 NVIDIA 显卡就用 CUDA，没有就用 CPU。本实验很小，在 CPU 上也只要几秒到几十秒
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| 设备:", device, "|",
      torch.cuda.get_device_name(0) if device.type == "cuda" else "没有检测到 CUDA，用 CPU 运行")

CHAPTER_NAME = "08_神经网络基础"
HOMEWORK_FILE = "作业_手写两层网络.py"


def load_homework():
    """载入本章的作业文件。设置了环境变量 COURSE_IMPL_DIR 时改从那个目录载入（维护课程时用）。"""
    here = Path.cwd().resolve()
    chapter = None
    for p in [here, *here.parents]:
        if p.name == CHAPTER_NAME:
            chapter = p
            break
        if (p / CHAPTER_NAME).is_dir():
            chapter = p / CHAPTER_NAME
            break
    if chapter is None:
        raise FileNotFoundError("请在课程目录内启动 Jupyter")
    base = chapter
    override = os.environ.get("COURSE_IMPL_DIR")
    if override:
        root = Path(override)
        if not root.is_absolute() and not (root / CHAPTER_NAME).is_dir():
            root = chapter.parent / root   # 相对路径按课程根目录算
        base = root / CHAPTER_NAME
    spec = importlib.util.spec_from_file_location("homework", base / HOMEWORK_FILE)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def attempt(fn, *args, **kwargs):
    """调用作业里的函数；还没实现时打印提示并返回 None，不中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {getattr(fn, '__name__', type(fn).__name__)}：{exc}")
        return None


hw = load_homework()
print("作业文件：", HOMEWORK_FILE)

In [ ]:
# 参数：改这里，然后从头运行
seed = 8
hidden_dim = 8
learning_rate = 0.05
steps = 250
torch.manual_seed(seed)
np.random.seed(seed)

In [ ]:
base_x = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
base_y = torch.tensor([0, 1, 1, 0])
# 数据先在 CPU 上生成（同一个种子在任何机器上都得到同样的点），再搬到 device
x = (base_x.repeat(32, 1) + 0.08 * torch.randn(128, 2)).to(device)
y = base_y.repeat(32).to(device)
print('x:', tuple(x.shape), x.dtype, x.device, 'y:', tuple(y.shape), y.dtype)

In [ ]:
def train_model(model, n_steps=steps):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for _ in range(n_steps):
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach()))
    return history

# 模型也是先在 CPU 上初始化，再搬到 device；模型和数据必须在同一个设备上
mlp = nn.Sequential(nn.Linear(2, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 2)).to(device)
history = train_model(mlp)
with torch.no_grad():
    accuracy = (mlp(x).argmax(1) == y).float().mean().item()
print(f'MLP final loss={history[-1]:.4f}, accuracy={accuracy:.3f}')

In [ ]:
grid_x, grid_y = np.meshgrid(np.linspace(-0.3, 1.3, 120), np.linspace(-0.3, 1.3, 120))
grid = torch.tensor(np.c_[grid_x.ravel(), grid_y.ravel()], dtype=torch.float32, device=device)
with torch.no_grad():
    region = mlp(grid).argmax(1).reshape(grid_x.shape).cpu().numpy()   # 画图前搬回 CPU
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(history); axes[0].set(title='训练损失', xlabel='步数', ylabel='交叉熵')
axes[1].contourf(grid_x, grid_y, region, alpha=0.3, levels=[-0.5, 0.5, 1.5])
axes[1].scatter(x[:, 0].cpu(), x[:, 1].cpu(), c=y.cpu(), s=12, cmap='coolwarm')
axes[1].set(title='MLP 的决策区域', xlim=(-0.3, 1.3), ylim=(-0.3, 1.3))
plt.tight_layout(); plt.show()

## 破坏实验：去掉隐藏层
线性分类器无法用一条直线分开 XOR。这里使用同一数据与训练步数作为对照。

In [ ]:
linear = nn.Linear(2, 2).to(device)
linear_history = train_model(linear)
with torch.no_grad():
    linear_accuracy = (linear(x).argmax(1) == y).float().mean().item()
print(f'linear final loss={linear_history[-1]:.4f}, accuracy={linear_accuracy:.3f}')
if accuracy > linear_accuracy:
    print('观察结论：非线性隐藏层改变了可表达的决策边界。')
else:
    # 默认参数下不会走到这里。改小 hidden_dim 或换 seed 之后可能出现，这是教程 8.1 节讲的现象，不是代码写错
    print('这一次 MLP 没有比线性模型好：隐藏层没有训练出来。能表示不等于训练得出来（教程 8.1 节），换一个 seed 或加大 hidden_dim 再跑。')

## 接上你的作业
把上面训练好的 MLP 的权重取出来，交给你在 `作业_手写两层网络.py` 里写的 NumPy 函数重算一遍，和 PyTorch 的结果对照。注意 `nn.Linear` 的 `weight` 是 `(输出维度, 输入维度)`，作业的约定相反，所以要转置。作业还没完成时这一格只打印提示。

In [ ]:
with torch.no_grad():
    w1 = mlp[0].weight.T.cpu().numpy().astype(np.float64)
    b1 = mlp[0].bias.cpu().numpy().astype(np.float64)
    w2 = mlp[2].weight.T.cpu().numpy().astype(np.float64)
    b2 = mlp[2].bias.cpu().numpy().astype(np.float64)
    torch_logits = mlp(x).cpu().numpy().astype(np.float64)
    torch_loss = float(nn.functional.cross_entropy(mlp(x), y))
x_np, y_np = x.cpu().numpy().astype(np.float64), y.cpu().numpy()

result = attempt(hw.two_layer_forward, x_np, w1, b1, w2, b2)
if result is not None:
    hidden, logits = result
    print('你的前向和 PyTorch 的 logits 最大相差:', float(np.abs(logits - torch_logits).max()), '（PyTorch 用 float32，差在 1e-5 以内算一致）')
    print('对全部样本都输出 0 的隐藏单元:', int((hidden.max(axis=0) == 0).sum()), '个，共', hidden.shape[1], '个')
my_loss = attempt(hw.cross_entropy_from_logits, torch_logits, y_np)
if my_loss is not None:
    print(f'你的交叉熵 {my_loss:.6f}，PyTorch 的交叉熵 {torch_loss:.6f}')